**Add a robot to Stage**

Start with a new Stage (File > New). To add a robot to the scene, copy-paste the following code snippet into the Script Editor and run it.

In [ ]:
import carb
from isaacsim.core.prims import Articulation
from isaacsim.core.utils.stage import add_reference_to_stage
from isaacsim.storage.native import get_assets_root_path
import numpy as np

assets_root_path = get_assets_root_path()
if assets_root_path is None:
    carb.log_error("Could not find Isaac Sim assets folder")
usd_path = assets_root_path + "/Isaac/Robots/FrankaRobotics/FrankaPanda/franka.usd"
prim_path = "/World/Arm"

add_reference_to_stage(usd_path=usd_path, prim_path=prim_path)
arm_handle = Articulation(prim_paths_expr=prim_path, name="Arm")
arm_handle.set_world_poses(positions=np.array([[0, -1, 0]]))

**Examine the robot**

Isaac Sim Core API has many function calls to retrieve information about the robot. Here are some examples for finding the number of joints and the joint names, various joint properties, and joint states.

Open a new tab in the Script Editor, copy-paste the following code snippet. This can only be run after the previous adding robot step, where arm_handle has already been established. Press Play before running the snippet. Physics must be running for these commands to work.

In [ ]:
# Get the number of joints
num_joints = arm_handle.num_joints
print("Number of joints: ", num_joints)

# Get joint names
joint_names = arm_handle.joint_names
print("Joint names: ", joint_names)

# Get joint limits
joint_limits = arm_handle.get_dof_limits()
print("Joint limits: ", joint_limits)

# Get joint positions
joint_positions = arm_handle.get_joint_positions()
print("Joint positions: ", joint_positions)

Notice when you pressed “Run”, it only prints the state once, even if the simulation is running. You’d have to keep pressing “Run” if you want to see more recent states. If you want to see the information printed at every physics step, you would need to insert these commands into a physics callback that runs at each physics step. We will go more in depth on how time stepping works in the next section Workflows.

To insert the commands into a physics callback, run the following snippet in a separate tab in the Script Editor.

In [ ]:
import asyncio
from isaacsim.core.api.simulation_context import SimulationContext

async def test():
    def print_state(dt):
        joint_positions = arm_handle.get_joint_positions()
        print("Joint positions: ", joint_positions)
    simulation_context = SimulationContext()
    await simulation_context.initialize_simulation_context_async()
    await simulation_context.reset_async()
    simulation_context.add_physics_callback("printing_state", print_state)

asyncio.ensure_future(test())

Start the simulation by pressing play, then run the snippet. You should see the information printed at every physics step into the terminal.

If printing at every physics step is no longer necessary, you can remove the physics callback by running the following snippet.

In [ ]:
simulation_context = SimulationContext()
simulation_context.remove_physics_callback("printing_state")

**Control the Robot**

There are many ways to control the robot in Isaac Sim. The lowest level is sending direct joint commands to set position, velocity, and efforts. Here is an example of how to control the robot using the Articulation API at the joint level.

Open a new tab in the Script Editor, copy-paste the following code snippet. This can only be run after the previous adding robot step, where arm_handle has already been established. Press Play before running the snippet. Physics must be running for these commands to work. We will provide two positions for you to toggle between. If you’ve added the print state snippet above to each physics step, you should be able to see the printed joints numbers change as the robot moves.

In [ ]:
# Set joint position randomly
arm_handle.set_joint_positions([[-1.5, 0.0, 0.0, -1.5, 0.0, 1.5, 0.5, 0.04, 0.04]])

In [ ]:
# Set all joints to 0
arm_handle.set_joint_positions([[0.0, 0.0, 0.0 , 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])

Similar to the get_joint_positions function above, the set_joint_positions here is only get executed once when you pressed “Run”. If you wish to send commands at every physics step, you would need to insert these commands into a physics callback that runs at each physics step.

**Save your work.**